# Coupled_Feedback — one-way coupling documentation

`MagneticCoupled` · ASN 28 validation form (TRUST-LiMe).



## Introduction

Documents the **one-way coupling** limitation of `force_lorentz`: the NS
velocity does **not** feed the magnetic convection.  With $\eta = 0$ and no
transport velocity, the magnetic field $\mathbf{b} = (y, -x)$ is an exact
equilibrium and must stay frozen — even when the NS velocity is non-zero.

## What is asserted

| # | Quantity | Formula |
|---|---|---|
| 1 | $\mathbf{b}$ stays frozen | $b_x(t) = y$, $b_y(t) = -x$ (machine precision) |
| 2 | hydrostatic balance holds | $\Delta p = -0.60$ (as in `MMS_Coupled`) |

The initial velocity $\mathbf{u}_0 = (0.1, 0)$ drives a transient flow that
decays; the magnetic field must **not** respond.  If the feedback were wired,
$\mathbf{b}$ would evolve.  This form is a **documentation test**: it pins the
current one-way behaviour so a future feedback implementation is noticed.



In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()



## Helpers: read a TRUST probe (`.son`) file


In [ ]:
import numpy as np
import os
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, sonde):
    """Return (time, values) for probe `sonde` of case `case`."""
    path = next((p for p in (f"{BUILD}/coarse/{case}_{sonde}.son",
                             f"{BUILD}/{case}_{sonde}.son") if os.path.exists(p)),
                f"{BUILD}/{case}_{sonde}.son")
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

print("BUILD =", BUILD)



## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`).


In [ ]:
from trustutils import run

NB_PROCS = 2

run.reset()
# --- mesh-convergence scan (shared ladder) ----------------------------------
# The single dataset src/coupled_feedback.data carries $nx/$ny node-count placeholders;
# every scale runs the SAME file via trustutils substitution
# (run.addCaseFromTemplate -> string.Template) into build/<scale>/ — one
# data-file name across scales:
#   coarse   = the baseline mesh
#   standard = cells x2 (h/2)
#   fine     = cells x4 (h/4)
# Each scale runs in its own build/<scale>/ directory so that the
# partitioned DOM_000*.Zones of one level never overwrite another's.
NX0 = NY0 = 11                        # baseline node counts (coarse)
meshes = {
    "coarse":   {"nx": 11, "ny": 11},            # the baseline mesh
    "standard": {"nx": 2*NX0-1, "ny": 2*NY0-1},
    "fine":     {"nx": 4*NX0-3, "ny": 4*NY0-3},
}
cases = {}
for lvl, dic in meshes.items():
    cases[lvl] = run.addCaseFromTemplate("coupled_feedback.data", lvl, dic=dic,
                                         nbProcs=NB_PROCS, excluNR=True)
    cases[lvl].partition()          # trust -partition coupled_feedback.data 2 -> PAR_coupled_feedback.data + DOM_000*.Zones
# --- merged two-way twin (the historical TwoWayFeedback case) --------------
# Same physical case as the baseline above (b = (y,-x), eta = 0, Lorentz
# hydrostatic balance) but with vitesse_hydraulique: the SOLVED NS velocity
# feeds the magnetic convection. Runs in the same form folder (relaxed R4:
# the datasets of the form live in its own src/).
case_twoway = run.addCase(".", "twoway_feedback.data", nbProcs=NB_PROCS)
case_twoway.partition()    # trust -partition twoway_feedback.data 2
run.printCases()
run.runCases()
case = cases["coarse"]            # alias kept for the read-back cells


## Checks

### #1 — magnetic field stays frozen (one-way coupling)


In [ ]:
t_bx, bx = read_son(case.dataFileName_, "SONDE_BX")
_, by = read_son(case.dataFileName_, "SONDE_BY")
# b = (y, -x): at the probe (0.35, 0.35), bx = 0.35, by = -0.35
bx_expected, by_expected = 0.35, -0.35
err_bx = float(np.max(np.abs(bx - bx_expected)))
err_by = float(np.max(np.abs(by - by_expected)))
print(f"max |bx - y|   = {err_bx:.3e}  (expected 0, machine precision)")
print(f"max |by - (-x)| = {err_by:.3e}  (expected 0, machine precision)")
assert err_bx < 1e-10, f"bx evolved: max |bx - y| = {err_bx}"
assert err_by < 1e-10, f"by evolved: max |by + x| = {err_by}"
print("OK: b stays frozen — the NS velocity does not feed the magnetic convection.")



### #2 — hydrostatic balance (as in MMS_Coupled)


In [ ]:
_, p1 = read_son(case.dataFileName_, "SONDE_P1")
_, p2 = read_son(case.dataFileName_, "SONDE_P2")
dp = p2[-1] - p1[-1]
expected = -(0.65**2 + 0.65**2 - 0.35**2 - 0.35**2)
err = abs(dp - expected) / abs(expected)
print(f"dp = {dp:.6g}   (analytic {expected:.4g}, err {err:.3e})")
assert dp < 0 and err < 0.05, f"pressure balance off: dp = {dp}"
print("OK: hydrostatic balance holds with non-zero initial velocity.")



## Mesh convergence

The shared refinement ladder: `coarse` = the baseline mesh above (`11×11`),
`standard` = cells ×2 (`21×21`), `fine` = cells ×4 (`41×41`).

Metric: the **Lorentz hydrostatic balance** error
$e_h = |\Delta p_h - \Delta p_{\mathrm{exact}}|$, where the exact pressure
drop $p = -(x^2+y^2)/\mu_0^2\rho$ is evaluated at each probe's
**effective** (cell-centre) position read from the `.son` header — under
refinement the probes snap to different cell centres, so the nominal
comparison would carry an $O(h)$ position error that is not a physics
error. The Green–Gauss $\mathrm{curl}\,b$ reconstruction is $O(h^2)$ at
the interior probes, and the observed order is $\log_2(e_k/e_{k+1})$.

In [ ]:
def probe_pos(path):
    """Effective (x, y) of a point probe, from the .son header."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("# Temps"):
                toks = line.replace("=", " = ").split()
                return (float(toks[toks.index("x") + 2]), float(toks[toks.index("y") + 2]))
    raise RuntimeError(f"no probe position header in {path}")

def read_son_dir(subdir, case_name, name):
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float), probe_pos(path)

def dp_error(subdir, case_name):
    """(abs err, exact drop) of the Lorentz hydrostatic pressure balance,
    with the exact drop evaluated at the probes' effective cell-centre
    positions (p = -(x^2+y^2)/mu0^2 rho, mu0 = rho = 1)."""
    _, p1, xy1 = read_son_dir(subdir, case_name, "SONDE_P1")
    _, p2, xy2 = read_son_dir(subdir, case_name, "SONDE_P2")
    dp_h = float(p2[-1] - p1[-1])
    dp_exact = -(xy2[0]**2 + xy2[1]**2) + (xy1[0]**2 + xy1[1]**2)
    return abs(dp_h - dp_exact), dp_exact

scan = {"coarse": ("coarse", case.dataFileName_),
        "standard": (f"standard", cases["standard"].dataFileName_),
        "fine": (f"fine", cases["fine"].dataFileName_)}
scan_nodes = {"coarse": (11, 11), "standard": (21, 21), "fine": (41, 41)}
res = {lvl: dp_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes      |dp - dp_exact(eff)|   rel      observed order")
prev = None
for lvl in ("coarse", "standard", "fine"):
    nx, ny = scan_nodes[lvl]
    e, dp_x = res[lvl]
    rel = e / abs(dp_x)
    order = "--"
    if prev is not None and prev > 0 and e > 0:
        order = f"{np.log2(prev / e):.2f}"
    print(f"{lvl:<9} {nx:>3}x{ny:<4} {e:<22.4e} {rel:<8.3f} {order}")
    prev = e
    # The baseline contract (dp within 5 pct of the exact drop) must hold at
    # EVERY level: the residual has a mesh-independent O(dt) floor (the
    # balance is reached only to temporal accuracy), so a strict monotone
    # decrease is not structural here.
    assert rel < 0.05, f"hydrostatic balance off at {lvl}: rel {rel:.3f}"
print("OK: hydrostatic Lorentz balance holds within 5 pct at every mesh level.")

## Two-way coupling — the solved velocity feeds the magnetic convection

The merged `twoway_feedback` case (historical `TwoWayFeedback` form):
same physics as the baseline above — $b = (y,-x)$, $\eta = 0$, and the
Lorentz force $-2(x,y)/\mu_0^2$ is a pure gradient, so the momentum
balance is the hydrostatic rest — but the magnetic equation carries the
**`vitesse_hydraulique`** keyword: its transporting velocity is the
*solved* NS velocity, bound in `completer()` once every equation is known
(the optional `Navier_Stokes_standard` equation is created only after
the magnetic block is read).

**One-way (the baseline case above)**: $\mathbf{b}$ stays frozen exactly —
the NS velocity does not feed the magnetic convection. **Two-way (this
case)**: the decaying initial flow $\mathbf{u}_0 = (0.1, 0)$ advects
$\mathbf{b}$ — $(\mathbf{u}\cdot\nabla)\mathbf{b} = (u_y, -u_x)$ — and the
probes must deviate from the frozen field.

| # | Quantity | Bound |
|---|:--|:--|
| 1 | velocity decays to the hydrostatic rest | $\max|\mathbf{u}|$ (last row) $< 10^{-3}$ |
| 2 | hydrostatic balance | $\Delta p = -0.60 \pm 5\%$ |
| 3 | two-way discriminator | $10^{-3} < \max_t \|\mathbf{b}-\mathbf{b}_0\| < 0.5$ |

In [ ]:
CASE2 = case_twoway.dataFileName_   # PAR_-prefixed after partition()

# 1) the velocity decays to the Lorentz hydrostatic rest
t_u, u = read_son(CASE2, "SONDE_U")
u_last = float(np.abs(u[-1]).max())
print(f"two-way: max |u| at t={t_u[-1]:.2f}: {u_last:.3e}  (started from 0.1)")
assert u_last < 1e-3, f"u did not decay: max |u| = {u_last}"

# 2) hydrostatic balance (same control as the one-way baseline)
_, p1 = read_son(CASE2, "SONDE_P1")
_, p2 = read_son(CASE2, "SONDE_P2")
dp = float(p2[-1] - p1[-1])
expected = -(0.65**2 + 0.65**2 - 0.35**2 - 0.35**2)
err = abs(dp - expected) / abs(expected)
print(f"two-way: dp = {dp:.6g}   (analytic {expected:.4g}, err {err:.3e})")
assert dp < 0 and err < 0.05, f"pressure balance off: dp = {dp}"

# 3) the two-way discriminator: b deviates from the frozen (y, -x) field
_, bx = read_son(CASE2, "SONDE_BX")
_, by = read_son(CASE2, "SONDE_BY")
dev = np.hypot(bx - 0.35, by + 0.35)
print(f"two-way: max deviation from the frozen field = {dev.max():.4e} at t = {t_u[int(np.argmax(dev))]:.2f}")
assert dev.max() > 1e-3, \
    f"b stayed frozen: the NS velocity does not feed the magnetic convection ({dev.max():.2e})"
assert dev.max() < 0.5, f"b deviation out of sanity range: {dev.max()}"
print("OK: the two-way coupling is live — b is advected by the solved velocity.")

## Computer Performance


In [ ]:
run.tablePerf()

